## Installs and Imports

In [ ]:
!pip install -U torch torchvision
!pip install datasets transformers pandas

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoImageProcessor, ResNetForImageClassification, AutoModelForImageClassification 
from datasets import load_dataset, load_from_disk
import copy
import torch.nn.functional as F
import os

## Dataset Prep

In [ ]:
dataset = load_dataset("uoft-cs/cifar10")
split = dataset["train"].train_test_split(test_size=0.2, seed=66)

train, val, test = split["train"], split["test"], dataset["test"]

In [ ]:
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")

In [ ]:
def transform_batch(batch):
    images = batch["img"]
    batch["pixel_values"] = processor(images, return_tensors="np")["pixel_values"]
    return batch

train = train.map(transform_batch, batched=True, batch_size=64)
val = val.map(transform_batch, batched=True, batch_size=64)
test = val.map(transform_batch, batched=True, batch_size=64)

In [ ]:
train.save_to_disk("/workspace/preprocessed/Cifar-10/train_converted")
val.save_to_disk("/workspace/preprocessed/Cifar-10/val_converted")
test.save_to_disk("/workspace/preprocessed/Cifar-10/test_converted")

In [ ]:
train = load_from_disk("/workspace/preprocessed/Cifar-10/train_converted")
val = load_from_disk("/workspace/preprocessed/Cifar-10/val_converted")
test = load_from_disk("/workspace/preprocessed/Cifar-10/test_converted")

## Model Prep

In [ ]:
device = "cuda" if torch.cuda_is_available() else "cpu"

In [ ]:
def leastSquares(Z0, Z1):
    W, residuals, rank, s = np.linalg.lstsq(Z0, Z1)
    return W, residuals

def cosineSimilarity(fine, aug):
    # Prevent NaNs
    eps = 1e-8
    out_aug = F.normalize(aug, dim=1, eps=eps)
    out_fine = F.normalize(fine, dim=1, eps=eps)

    cos_sim = (out_aug * out_fine).sum(dim=1).mean().item()
    return cos_sim

In [ ]:
# ResNet50 HF GitHub: https://github.com/huggingface/transformers/blob/main/src/transformers/models/resnet/modeling_resnet.py
# Uses Global Average Pooling (GAP) instead of CLS Tokens like ViT. GAP is only applied at the end, whereas ViT is always present. 
# Look at lines #315 -> #270 -> #206 in the GitHub
class Hooks(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.GAP = []
    
    def forward(self, pixel_values):
        x = self.model.embedder(pixel_values)
        for i, stage_module in enumerate(self.model.encoder.stages):
            x = stage_module(x)
            self.GAP.append(x)
        
        pooled_output = self.model.pooler(self.GAP[-1]) # Shape: (batch_size, hidden_dim); Used for final comparison
        logits = self.model.classifier(pooled_output) # Shape: (batch_size, num_classes)
        return self.GAP

In [ ]:
refer = ResNetForImageClassification.from_pretrained("microsoft/resnet-50").to(device)
base_H = Hooks(copy.deepcopy(refer)).to(device)
fine_tuned_H = Hooks(AutoModelForImageClassification.from_pretrained("yhyan/resnet-50-finetuned-eurosat")).to(device) # https://huggingface.co/yhyan/resnet-50-finetuned-eurosat

In [ ]:
train.set_format(type="torch", columns=["label", "img", "pixel_values"])
val.set_format(type="torch", columns=["label", "img", "pixel_values"])
test.set_format(type="torch", columns=["label", "img", "pixel_values"])

def clip_collate_fn(batch):
    # images = np.stack([example["pixel_values"] for example in batch])
    # images = torch.from_numpy(images)
    # labels = torch.tensor([example["label"] for example in batch])
    images = torch.stack([example["pixel_values"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch])
    
    return {
        "pixel_values": images.to(device),
        "labels": labels.to(device)
    }

train_loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
val_loader = DataLoader(val, batch_size=64, shuffle=False, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
test_loader = DataLoader(test, batch_size=64, shuffle=False, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

In [ ]:
# Extracting Embeddings
indices = [i for i in range(0, 50, 4)].extend([49]) # 14 total. 

Z0 = {i: i for i in indices}

Z1_last_lsr = []

with torch.no_grad():
    for batch in tqdm(train_loader, desc=f"Extracting Train Dataset Vectors"):
        images = batch["pixel_values"].to(device, non_blocking=True)

        out_base = base_H(images)
        out_fine_tuned = fine_tuned_H(images)
        
        for i in indices:
            Z0[i].append(out_base[i].float().cpu())

        Z1_last_lsr.append(out_fine_tuned[49].float().cpu())

Z1_last_lsr = torch.cat(Z1_last_lsr)
Z1_last_lsr = Z1_last_lsr.cpu().numpy()

In [ ]:
W = {}
resid = {}

for key, value in Z0.items():
    value = torch.cat(value)
    value = value.cpu().numpy()
    W[key], resid[key] = leastSquares(value, Z1_last_lsr)

In [ ]:
class Augmented(torch.nn.Module):
    def __init__(self, model, classifier=None, W=None, transform_stage=-1):
        super().__init__()
        self.model = model
        self.classifier = classifier if classifier is not None else torch.nn.Linear(in_features=2048, out_features=10)
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_stage = transform_stage

    def forward(self, pixel_values):
        x = self.model.embedder(pixel_values)
        for i, stage_module in enumerate(self.model.encoder.stages):
            x = stage_module(x)
            if i == self.transform_stage:
                x = x @ self.W
                manipulated_raw = x
                break
            if i == 49:
                manipulated_raw = x
        
        pooled_output = self.model.pooler(x) # Shape: (batch_size, hidden_dim); Used for final comparison
        logits = self.classifier(pooled_output) # Shape: (batch_size, num_classes)
        return logits, manipulated_raw

In [ ]:
# Augmenting Models
aug = {}

for i in indices: 
    model = Augmented(copy.deepcopy(refer), W=W[i], transform_stage=i)
    model = model.eval().to(device)
    aug[i] = model

In [ ]:
base = Augmented(copy.deepcopy(refer)).to(device)
f_t = AutoModelForImageClassification.from_pretrained("yhyan/resnet-50-finetuned-eurosat")
fine_tuned = Augmented(copy.deepcopy(f_t), classifier=f_t.classifier).to(device)

In [ ]:
correct_base = 0
correct_fine_tuned = 0
total_samples = 0

correct = {}
sim_cls = {}
for i in indices:
    correct[i] = 0
    sim_cls[i] = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        total_samples += labels.size(0)

        # Base Model
        logits_base, manipulated_raw_base = base(images)
        preds = logits_base.argmax(dim=1)
        correct_base += (preds == labels).sum().item()

        # Fine-Tuned Model
        logits_fine_tuned, manipulated_raw_fine_tuned = fine_tuned(images)
        preds = logits_fine_tuned.argmax(dim=1)
        correct_fine_tuned += (preds == labels).sum().item()

        # Augmented Base Models
        for i in range(12):
            logits_augmented, manipulated_raw_augmented = aug[i](images)
            preds = logits_augmented.argmax(dim=1)
            correct[i] += (preds == labels).sum().item()
            sim_cls[i].append(cosineSimilarity(manipulated_raw_fine_tuned, manipulated_raw_augmented))

In [ ]:
print(f"Augmented ResNet50 on Entire Cifar-10 Results")
for i in indices:
    print(f"\tAugmented {i+1} - Last (49) Layer Accuracy: {correct[i]}")
    print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i+1} Layer: {sim_cls[i]:.4f}")
print(f"Base Accuracy: {correct_base:.4f}")
print(f"Fine-Tuned Accuracy: {correct_fine_tuned:.4f}")

In [ ]:
folder = f"./Entire_{type}"
os.makedirs(folder, exist_ok=True)

train_size = len(train_loader)

data = {
    'Train_Data_Size': [train_size]*len(indices),
    "Transformation": [type if type != "" else "None"] * len(indices),
    'W': [W[i] for i in indices], # data.loc[0, "W"] -> First layer transformation
    'Accuracy': [correct[i] for i in indices], 
    "Co_Sim_CLS": [sim_cls[i] for i in indices],
}

df = pd.DataFrame(data, index=indices)

name = f"{type}_Entire_Size_{train_size}_Augmentation_Results.csv"
path = os.path.join(folder, name)
df.to_csv(path)

In [ ]:
torch.cuda.empty_cache()